# Accelerator Orbit Alignment Hackathon: Beam Orbit Benchmark
## Master Reference (with Weights & Biases / W&B Leaderboard Integration)

**Tutorial & Challenge Context:**
* **Venue:** Korea Multi-purpose Accelerator Complex (KOMAC) / KAERI (Tentative)
* **Target Audience:** Graduate Students & Early-Career Postdoctoral Researchers
* **Curricular Alignment:** Lecture 1 (Chong Shik Park - AI/ML & Surrogate Modeling, 80 min) & Lecture 2 (Thorsten Hellert - Agentic AI & Scientific Workflows, 80 min)
* **Challenge Architecture (Gymnasium + W&B Benchmark):**
  1. **Standardized Gymnasium Environment:** `AcceleratorOrbitGymEnv` matching the standard benchmark specification (continuous `Box` observation & action spaces).
  2. **Baseline Track 0:** Classical Linear Orbit Response Matrix (ORM) SVD Pseudo-Inverse Controller.
  3. **Baseline Track 1:** Supervised Neural Digital Twin (`MLPRegressor`) + SciPy Bounded Inverse Optimization.
  4. **Challenge Track 2:** Closed-Loop Reinforcement Learning Policy Agent (`act(obs, info)`).
  5. **Live W&B Scoring & Leaderboard:** Automated benchmark evaluation across 30 fixed test seeds with live telemetry logging to [Weights & Biases (wandb.ai)](https://wandb.ai).


In [ ]:
# Step 0: Imports, Environment Setup, and Weights & Biases (W&B) Integration
import time
import numpy as np
import scipy.optimize as opt
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import root_mean_squared_error

# Fix random seed for reproducible benchmark evaluations
np.random.seed(42)

# Optional: Install and log into Weights & Biases (wandb) for the live challenge leaderboard
# Uncomment the line below if running in Google Colab or a fresh virtual environment:
# !pip install -q wandb

try:
    import wandb
    wandb.login()
    WANDB_AVAILABLE = True
    print("Weights & Biases (wandb) imported successfully. Live leaderboard logging ENABLED.")
except ImportError:
    WANDB_AVAILABLE = False
    print("Weights & Biases not installed. Running in local benchmark mode.")
    print("  (Tip: Run '!pip install wandb' to log scores to the live challenge dashboard!)")

# W&B Challenge Configuration
WANDB_PROJECT = "accelerator-orbit-hackathon"  # Shared workshop project
WANDB_ENTITY = None                            # Workshop entity or team workspace (or None)
TEAM_NAME = "Team_Alpha"                       # Change to your team's assigned name!
print(f"Active Team: {TEAM_NAME} | Target Project: {WANDB_PROJECT}")


/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


## 1. Accelerator Lattice Physics & 2D Schematic

The challenge lattice consists of a beam transport section with:
* **Actuators ($X$):** 6 dipole correctors (3 Horizontal HCM, 3 Vertical VCM) with hardware bounds $[-5.0\text{ A}, +5.0\text{ A}]$.
* **Optics:** Alternating quadrupole magnets (QF focusing, QD defocusing) providing transverse betatron containment.
* **Diagnostics ($Y$):** 12 Beam Position Monitors (BPM1 to BPM12) measuring transverse beam offsets in mm.


In [ ]:
# Step 1: 2D Schematic Diagram of the Accelerator Lattice
# (Full schematic generator code is modularized in src.visualization.draw_beamline)
import os
from IPython.display import Image, display

img_path = "img/beamline.png" if os.path.exists("img/beamline.png") else "../img/beamline.png"
display(Image(filename=img_path))


## 2. Standard Gymnasium Environment (`AcceleratorOrbitGymEnv`)

Following the **Hackathon (Reinforcement Learning for Autonomous Accelerators)** challenge format, the accelerator simulator is encapsulated in a Gymnasium-compatible environment:
* **Observation Space:** `Box(-10.0, +10.0, shape=(12,), dtype=np.float32)` representing the 12 BPM readings (in mm).
* **Action Space:** `Box(-0.8, +0.8, shape=(6,), dtype=np.float32)` representing per-step incremental kick adjustments $\Delta I_c$ (in Amperes).
* **Hardware Constraints:** Hard power supply current limits $[-5.0\text{ A}, +5.0\text{ A}]$.
* **Reward Function:** $r_t = -10.0 \times \text{RMS}(Y_t) - 0.01 \|\Delta I_{c,t}\|_2^2 + (50.0\text{ if success else }0.0)$.
* **Termination Criteria:** Centered when $\text{RMS} < 0.030\text{ mm}$ (Success) or `current_step >= max_steps` (12 steps truncated).


In [ ]:
# Step 2: Define Standardized Hackathon Gymnasium Environment
class BoxSpace:
    """Gymnasium Box space fallback ensuring zero-dependency compatibility."""
    def __init__(self, low, high, shape, dtype=np.float32):
        self.low = np.full(shape, low, dtype=dtype)
        self.high = np.full(shape, high, dtype=dtype)
        self.shape = shape
        self.dtype = dtype
    def sample(self):
        return np.random.uniform(self.low, self.high).astype(self.dtype)

class AcceleratorOrbitGymEnv:
    """Accelerator Orbit Hackathon: Multi-Corrector Beam Alignment Environment."""
    metadata = {"render_modes": ["human"]}

    def __init__(self, max_steps=12, target_rms=0.030, noise_std=0.010):
        self.n_correctors = 6
        self.n_bpms = 12
        self.max_steps = max_steps
        self.target_rms = target_rms
        self.noise_std = noise_std

        self.action_space = BoxSpace(low=-0.8, high=0.8, shape=(self.n_correctors,))
        self.observation_space = BoxSpace(low=-10.0, high=10.0, shape=(self.n_bpms,))

        # Physical Transfer Response Matrix R
        s_bpm = np.linspace(1.0, 15.0, self.n_bpms)
        s_corr = np.linspace(0.2, 8.0, self.n_correctors)
        self.R = np.zeros((self.n_correctors, self.n_bpms))
        for i, sc in enumerate(s_corr):
            for j, sb in enumerate(s_bpm):
                if sb > sc:
                    phase = 2.0 * np.pi * 0.18 * (sb - sc)
                    self.R[i, j] = 1.2 * np.sin(phase) / (1.0 + 0.05 * (sb - sc))

        self.I_corr = np.zeros(self.n_correctors, dtype=np.float32)
        self.perturbation = np.zeros(self.n_bpms, dtype=np.float32)
        self.current_step = 0
        self.np_random = np.random.RandomState(42)

    def reset(self, seed=None, options=None):
        if seed is not None:
            self.np_random = np.random.RandomState(seed)
        self.current_step = 0
        self.I_corr = np.zeros(self.n_correctors, dtype=np.float32)

        # Realistic upstream steering misalignment drift perturbation
        drift = self.np_random.uniform(-1.2, 1.2, size=2)
        self.perturbation = (drift[0] * self.R[0] + drift[1] * self.R[1]).astype(np.float32)

        obs = self._get_obs()
        init_rms = float(np.sqrt(np.mean(obs**2)))
        return obs, {"initial_rms": init_rms, "step": 0, "currents": self.I_corr.copy()}

    def _get_obs(self):
        # Cubic iron magnetic saturation: B(I) ~ I - 0.015 * I^3
        response = (self.I_corr @ self.R) - 0.015 * (self.I_corr**3) @ self.R
        noise = self.np_random.normal(0, self.noise_std, size=self.n_bpms)
        return (self.perturbation + response + noise).astype(np.float32)

    def step(self, action):
        action = np.clip(action, self.action_space.low, self.action_space.high).astype(np.float32)
        self.I_corr = np.clip(self.I_corr + action, -5.0, 5.0)
        self.current_step += 1

        obs = self._get_obs()
        rms = float(np.sqrt(np.mean(obs**2)))

        # Step Reward: Negative distance penalty + step action effort
        reward = -10.0 * rms - 0.01 * float(np.sum(action**2))
        terminated = (rms < self.target_rms)
        truncated = (self.current_step >= self.max_steps)

        if terminated:
            reward += 50.0  # Success completion bonus

        info = {
            "rms": rms,
            "step": self.current_step,
            "is_success": terminated,
            "currents": self.I_corr.copy()
        }
        return obs, reward, terminated, truncated, info

env = AcceleratorOrbitGymEnv()
print("Hackathon Environment instantiated successfully.")


## 3. Supervised Forward Modeling: Linear vs. Neural Surrogate

In modern accelerator operations, numerical tracking simulation datasets are converted into ultra-fast surrogate models (Lecture 1).
Here we generate 1,500 simulation shots (80% train, 20% test) to train an MLP Neural Digital Twin.


In [ ]:
# Step 3: Offline Data Generation & Surrogate Training
n_train_samples = 1500
X_offline = np.random.uniform(-3.0, 3.0, size=(n_train_samples, env.n_correctors))
Y_linear = X_offline @ env.R
Y_sat = - 0.015 * (X_offline**3) @ env.R
bpm_noise = np.random.normal(0, 0.010, size=(n_train_samples, env.n_bpms))
Y_offline = Y_linear + Y_sat + bpm_noise

X_train, X_test, Y_train, Y_test = train_test_split(X_offline, Y_offline, test_size=0.20, random_state=42)

linear_model = Ridge(alpha=1.0).fit(X_train, Y_train)
rmse_linear = root_mean_squared_error(Y_test, linear_model.predict(X_test))

surrogate_model = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    solver='adam',
    alpha=1e-4,
    max_iter=400,
    random_state=42
).fit(X_train, Y_train)
rmse_mlp = root_mean_squared_error(Y_test, surrogate_model.predict(X_test))

print("="*55)
print(f"OFFLINE SURROGATE EVALUATION (Holdout 20%):")
print(f"  Linear Ridge Baseline RMSE: {rmse_linear:.4f} mm")
print(f"  MLP Neural Surrogate RMSE:  {rmse_mlp:.4f} mm")
print("="*55)


---
## 4. The Three Benchmark Competitor Controllers

Following the Hackathon benchmark framework, every controller implements a unified agent interface:
`def act(self, obs: np.ndarray, info: dict) -> np.ndarray`

1. **Baseline 0 (Classical Linear ORM):** Standard accelerator control using Moore-Penrose pseudo-inverse $\Delta I_c = - \alpha R^+ Y$.
2. **Baseline 1 (Supervised Digital Twin Inversion):** Model-based batch inversion using `scipy.optimize.minimize` (L-BFGS-B).
3. **Challenge Track 2 (Closed-Loop RL Agent):** Interactive policy agent with physics-informed prior and dynamic damping.


In [ ]:
# Step 4: Define Benchmark Controllers

# 1. Baseline 0: Classical Linear ORM Pseudo-Inverse Controller
class ClassicalSVDController:
    """Baseline 0: Classical linear Orbit Response Matrix pseudo-inverse feedback."""
    def __init__(self, R_matrix, gain=0.45):
        self.gain = gain
        self.R_pinv = np.linalg.pinv(R_matrix.T)

    def act(self, obs, info=None):
        return np.clip(- self.gain * (self.R_pinv @ obs), -0.8, 0.8)

# 2. Baseline 1: Supervised Forward Surrogate + SciPy Inversion
class SupervisedMLPController:
    """Baseline 1: Model-based batch optimizer using the MLP digital twin."""
    def __init__(self, surrogate):
        self.surrogate = surrogate

    def act(self, obs, info=None):
        def objective(delta_I):
            pred = self.surrogate.predict(delta_I.reshape(1, -1))[0]
            res = obs + pred
            return np.sum(res**2) + 0.0001 * np.sum(delta_I**2)
        res = opt.minimize(objective, np.zeros(6), bounds=[(-5.0, 5.0)]*6, method='L-BFGS-B')
        return np.clip(res.x, -0.8, 0.8)

# 3. Track 2: Closed-Loop RL Policy Agent with Physics Prior
class RLOrbitChallengeAgent:
    """Track 2: Closed-loop policy agent for orbit alignment hackathon."""
    def __init__(self, R_matrix, gain=0.45):
        self.base_R_pinv = np.linalg.pinv(R_matrix.T)
        self.gain = gain
        self.W_tune = np.zeros((6, 12))
        self.bias = np.zeros(6)

    def act(self, obs, info=None):
        step_idx = info.get("step", 0) if info else 0
        action_phys = - self.gain * (self.base_R_pinv @ obs)
        action_learned = self.W_tune @ obs + self.bias
        action = action_phys + 0.05 * np.tanh(action_learned)
        damping = 1.0 / (1.0 + 0.03 * step_idx)
        return np.clip(action * damping, -0.8, 0.8)

b0 = ClassicalSVDController(env.R)
b1 = SupervisedMLPController(surrogate_model)
b2 = RLOrbitChallengeAgent(env.R)
print("All three competitor controllers initialized.")


---
## 5. Standardized Benchmark Evaluation & Live W&B Scoring

To ensure scientific rigor, agents are evaluated across **30 standardized test seeds** with diverse physical drift perturbations.
The **Official Hackathon Benchmark Score** combines:
$$\text{Score} = 10 \times \text{Success Rate (\%)} - 2 \times \bar{T} + 20 \times \left( \frac{0.03}{\max(\overline{\text{RMS}}, 0.005)} \right) - 2 \times \|\mathbf{I}\|_2$$

When enabled, telemetry, seed breakdowns, and scores are automatically uploaded to your **Weights & Biases (wandb)** dashboard for real-time leaderboard ranking!


In [ ]:
# Step 5: Standardized Evaluation Harness with Weights & Biases Logging
def evaluate_hackathon_benchmark(
    agent,
    agent_name="Agent",
    team_name=TEAM_NAME,
    seeds=range(1, 31),
    use_wandb=True,
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY
):
    n_seeds = len(seeds)
    successes = 0
    total_steps = 0
    final_rms_list = []
    current_norms = []
    latencies = []
    table_rows = []

    for seed in seeds:
        obs, info = env.reset(seed=seed)
        terminated = False
        truncated = False
        step = 0
        t0 = time.time()

        while not (terminated or truncated):
            action = agent.act(obs, info)
            obs, reward, terminated, truncated, info = env.step(action)
            step += 1

        elapsed_ms = (time.time() - t0) * 1000.0
        latencies.append(elapsed_ms)

        is_succ = info.get("is_success", False)
        if is_succ:
            successes += 1
        total_steps += step
        final_rms_list.append(info["rms"])
        norm_I = float(np.linalg.norm(info["currents"]))
        current_norms.append(norm_I)

        table_rows.append([seed, info["initial_rms"], info["rms"], step, is_succ, norm_I, elapsed_ms])

    success_rate = (successes / n_seeds) * 100.0
    mean_steps = total_steps / n_seeds
    mean_rms = float(np.mean(final_rms_list))
    mean_norm = float(np.mean(current_norms))
    mean_latency = float(np.mean(latencies))

    # Official Orbit Alignment Hackathon Scoring Formula
    score = success_rate * 10.0 - 2.0 * mean_steps + (0.03 / max(mean_rms, 0.005)) * 20.0 - 2.0 * mean_norm

    # --------------------------------------------------------------------------
    # Weights & Biases (W&B) Logging
    # --------------------------------------------------------------------------
    if use_wandb and WANDB_AVAILABLE:
        try:
            run_name = f"{team_name}_{agent_name}"
            run = wandb.init(
                project=project,
                entity=entity,
                name=run_name,
                tags=["orbit-hackathon", agent_name, team_name],
                config={
                    "team": team_name,
                    "agent": agent_name,
                    "n_seeds": n_seeds,
                    "target_rms_mm": env.target_rms,
                    "max_steps": env.max_steps,
                    "noise_std_mm": env.noise_std
                },
                reinit=True
            )

            # Log summary metrics to run.summary (displays on W&B Leaderboard table)
            run.summary["challenge_score"] = float(round(score, 1))
            run.summary["success_rate_pct"] = float(round(success_rate, 1))
            run.summary["mean_steps"] = float(round(mean_steps, 2))
            run.summary["mean_final_rms_mm"] = float(round(mean_rms, 4))
            run.summary["mean_current_norm_A"] = float(round(mean_norm, 2))
            run.summary["mean_latency_ms"] = float(round(mean_latency, 2))

            # Log full seed breakdown table
            wandb_cols = ["Seed", "Initial_RMS_mm", "Final_RMS_mm", "Steps", "Success", "Current_Norm_A", "Latency_ms"]
            wandb_table = wandb.Table(columns=wandb_cols, data=table_rows)
            wandb.log({"seed_breakdown_table": wandb_table})

            print(f"[W&B] Logged '{run_name}' to {run.url} (Score: {score:.1f} pts)")
            run.finish()
        except Exception as e:
            print(f"[W&B Warning] Logging skipped: {e}")

    return {
        "success_rate": success_rate,
        "mean_steps": mean_steps,
        "mean_rms": mean_rms,
        "mean_norm": mean_norm,
        "mean_latency": mean_latency,
        "score": score
    }

print("Running Official Hackathon Benchmark across 30 Test Seeds...")
res_b0 = evaluate_hackathon_benchmark(b0, agent_name="Baseline0_LinearSVD")
res_b1 = evaluate_hackathon_benchmark(b1, agent_name="Baseline1_SupervisedMLP")
res_b2 = evaluate_hackathon_benchmark(b2, agent_name="Track2_RLAgent")

print("="*85)
print("                        OFFICIAL HACKATHON BENCHMARK LEADERBOARD")
print("="*85)
print(f"{'Rank':<5} | {'Agent / Controller Name':<28} | {'Success':<8} | {'Steps':<6} | {'Final RMS':<10} | {'Score':<8}")
print("-"*85)
leaderboard = [
    ("Baseline 1 (Supervised MLP Inversion)", res_b1),
    ("Baseline 0 (Linear ORM SVD Controller)", res_b0),
    ("Track 2 (RL Closed-Loop Agent)", res_b2),
]
leaderboard.sort(key=lambda x: x[1]["score"], reverse=True)

for rank, (name, metrics) in enumerate(leaderboard, 1):
    print(f"#{rank:<4} | {name:<28} | {metrics['success_rate']:>6.1f}% | {metrics['mean_steps']:>5.1f} | {metrics['mean_rms']:>8.4f}mm | {metrics['score']:>7.1f}")
print("="*85)


---
## 6. Visualizing Trajectory Alignment & Dynamic Convergence


In [ ]:
# Step 6: Generate Dual Trajectory Overlay & Dynamic Convergence Curves
# Evaluate on canonical test seed 42
obs_canon, info_canon = env.reset(seed=42)
initial_canon_rms = info_canon["initial_rms"]

# Run single episode for each controller
trajectories = {}
for name, agent in [("Linear SVD", b0), ("Supervised MLP", b1), ("RL Agent", b2)]:
    obs, info = env.reset(seed=42)
    rms_hist = [info["initial_rms"]]
    while not (info.get("is_success", False) or info.get("step", 0) >= env.max_steps):
        action = agent.act(obs, info)
        obs, reward, term, trunc, info = env.step(action)
        rms_hist.append(info["rms"])
        if term or trunc:
            break
    trajectories[name] = (obs, rms_hist)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.8))

# Subplot 1: Beamline Trajectory Alignment
bpm_stations = range(1, env.n_bpms + 1)
ax1.plot(bpm_stations, obs_canon, 'ro-', label=f'Uncorrected Drift (RMS={initial_canon_rms:.3f} mm)', linewidth=1.8)
ax1.plot(bpm_stations, trajectories["Linear SVD"][0], 'c^--', label=f'Linear SVD (RMS={trajectories["Linear SVD"][1][-1]:.3f} mm)', linewidth=2.0)
ax1.plot(bpm_stations, trajectories["Supervised MLP"][0], 'b^--', label=f'Supervised MLP (RMS={trajectories["Supervised MLP"][1][-1]:.3f} mm)', linewidth=2.0)
ax1.plot(bpm_stations, trajectories["RL Agent"][0], 'gs-', label=f'RL Agent (RMS={trajectories["RL Agent"][1][-1]:.3f} mm)', linewidth=2.2)
ax1.axhline(0, color='black', linestyle=':', alpha=0.6)
ax1.set_title("BPM Trajectory Alignment: Uncorrected vs. Competitors", fontsize=11, fontweight='bold')
ax1.set_xlabel("BPM Station Index", fontsize=10)
ax1.set_ylabel("Beam Displacement (mm)", fontsize=10)
ax1.legend(frameon=True, fontsize=8.5)
ax1.grid(True, linestyle='--', alpha=0.4)

# Subplot 2: Dynamic Convergence over Steps
for name, col, marker in [("Linear SVD", "cyan", "^"), ("Supervised MLP", "blue", "o"), ("RL Agent", "green", "s")]:
    hist = trajectories[name][1]
    ax2.plot(range(len(hist)), hist, f'{col}{marker}-', linewidth=2.0, label=f'{name} (Final: {hist[-1]:.3f} mm)')

ax2.axhline(env.target_rms, color='red', linestyle='--', linewidth=1.5, label=f'Target Tolerance ({env.target_rms:.3f} mm)')
ax2.set_title("Convergence Dynamics across Interaction Steps", fontsize=11, fontweight='bold')
ax2.set_xlabel("Control Step $t$", fontsize=10)
ax2.set_ylabel("Beam Orbit RMS (mm)", fontsize=10)
ax2.legend(frameon=True, fontsize=8.5)
ax2.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()


---
## 7. Final Team Showcase Cell (16:30–17:00)
Run this single summary cell during team presentations to display your official Hackathon submission score!


In [ ]:
print("="*68)
print("          ACCELERATOR ORBIT HACKATHON: TEAM SHOWCASE")
print("="*68)
print(f"1. ENVIRONMENT SPEC:        AcceleratorOrbitGymEnv (12 BPMs, 6 Steerers)")
print(f"2. EVALUATION SUITE:        30 Standardized Fixed Test Seeds")
print(f"3. TOP CONTROLLER:          {leaderboard[0][0]}")
print(f"4. CHALLENGE BENCHMARK:     Success: {leaderboard[0][1]['success_rate']:.1f}% | Mean Steps: {leaderboard[0][1]['mean_steps']:.1f}")
print(f"5. RESIDUAL ORBIT RMS:      {leaderboard[0][1]['mean_rms']:.4f} mm (< 0.03 mm Noise Floor)")
print(f"6. OFFICIAL HACKATHON SCORE:    {leaderboard[0][1]['score']:.1f} pts")
if WANDB_AVAILABLE:
    print(f"7. LIVE W&B DASHBOARD:      https://wandb.ai/{WANDB_ENTITY or '<your-workspace>'}/{WANDB_PROJECT}")
print("="*68)
